# UCS420 Cognitive Computing
## Assignment 9: Landing AI Vision Based Cognitive System

Name: Ansh Sharma  
Roll number: 1024170344

## Aim

In this assignment, I designed two vision based cognitive assistants using the LandingLens workflow. The first assistant helps a user read information from a medicine bottle. The second classifies visible facial expressions for an emotion-aware assistant. The notebook also checks the image folders before they are uploaded for training.

## 1. Medicine label reading assistant

### Problem

A user can take a photo of a medicine bottle and ask the assistant to find the printed label information. The vision model can locate the bottle and label area. An OCR step can then read the text, which the assistant can show back to the user for checking.

### Proposed workflow

1. Collect bottle photos from different angles, distances, and lighting conditions.
2. Create an object detection project in LandingLens.
3. Upload the images and draw boxes around `Bottle` and `Label`. Add an `Expiry_Date` box when the date is visible.
4. Train the model, then test it on new bottle images that were not used for training.
5. Send the detected label crop to an OCR tool and display the recognized text with the original image.
6. Let the user verify the medicine name and date before using the information.

The object detector finds where the text is. OCR reads the characters inside that region, so both steps are needed for label reading.

### Dataset folder check

I would keep the original bottle photos in `data/medicine_bottles`. The following cell counts supported image files and reports the image dimensions. Create the folder and add the photos before running it.

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

medicine_folder = Path('data/medicine_bottles')
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
medicine_images = sorted(path for path in medicine_folder.rglob('*') if path.suffix.lower() in image_extensions) if medicine_folder.exists() else []

medicine_summary = []
for image_path in medicine_images:
    try:
        with Image.open(image_path) as image:
            medicine_summary.append({
                'File': str(image_path),
                'Width': image.width,
                'Height': image.height,
                'Mode': image.mode
            })
    except Exception:
        medicine_summary.append({'File': str(image_path), 'Width': None, 'Height': None, 'Mode': 'Could not open'})

if medicine_summary:
    display(pd.DataFrame(medicine_summary))
else:
    print('No medicine bottle images found. Add JPG, PNG, BMP, or WEBP files to data/medicine_bottles.')

In [ ]:
import matplotlib.pyplot as plt

preview_images = medicine_images[:6]
if preview_images:
    figure, axes = plt.subplots(1, len(preview_images), figsize=(4 * len(preview_images), 4))
    if len(preview_images) == 1:
        axes = [axes]
    for axis, image_path in zip(axes, preview_images):
        with Image.open(image_path) as image:
            axis.imshow(image.convert('RGB'))
        axis.set_title(image_path.name)
        axis.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Image previews will appear here after the dataset is added.')

### LandingLens experiment

For this task, I would select **Object Detection** when creating the LandingLens project. I would label the images with bounding boxes and keep some images aside for testing. After training, I would use **Try Model** with new photos and check whether the boxes cover the bottle and label correctly. If the model misses small expiry dates, I would add more close-up examples with clear lighting and label them again. After the test looks reasonable, the model can be deployed as an endpoint for an application to call.

### Assistant response

The assistant would show the label crop and the text returned by OCR. It should ask the user to confirm the medicine name and expiry date when the image is blurry or the text is uncertain. It should not recommend a dose based only on a photo.

## 2. Emotion-aware cognitive assistant

### Problem

This assistant uses a camera image to classify a visible facial expression, such as happy, sad, angry, surprised, or neutral. The result can help the interface choose a considerate response, but a facial expression does not reliably tell us how a person feels. The output should be treated as a visual cue and the user should be able to correct it.

### Proposed workflow

1. Collect consent-based face images with varied lighting, camera angles, and backgrounds.
2. Use a **Classification** project in LandingLens and create one class for each expression label.
3. Upload the images into the matching classes and check that the examples are varied and correctly labeled.
4. Train the model and test it on separate images. Review the confusion matrix or class-level results if available.
5. Deploy the model only for an optional interface feature. Do not use the prediction to make decisions about a person.

I would use expression labels such as `Happy`, `Sad`, `Angry`, `Surprised`, and `Neutral`. These are labels for visible expressions, not confirmed emotional states.

### Dataset folder check

For the classification dataset, I would create one folder per expression label under `data/facial_expressions`. The folder name becomes the class label in this quick audit.

In [ ]:
emotion_folder = Path('data/facial_expressions')
emotion_classes = ['Happy', 'Sad', 'Angry', 'Surprised', 'Neutral']
emotion_summary = []

for class_name in emotion_classes:
    class_folder = emotion_folder / class_name
    class_images = [path for path in class_folder.rglob('*') if path.suffix.lower() in image_extensions] if class_folder.exists() else []
    emotion_summary.append({
        'Expression class': class_name,
        'Image count': len(class_images)
    })

emotion_table = pd.DataFrame(emotion_summary)
display(emotion_table)

if emotion_table['Image count'].sum() == 0:
    print('Add images to the matching class folders before uploading the dataset.')

In [ ]:
emotion_preview = []
for class_name in emotion_classes:
    class_folder = emotion_folder / class_name
    if class_folder.exists():
        class_images = [path for path in class_folder.rglob('*') if path.suffix.lower() in image_extensions]
        emotion_preview.extend((class_name, path) for path in class_images[:1])

if emotion_preview:
    figure, axes = plt.subplots(1, len(emotion_preview), figsize=(4 * len(emotion_preview), 4))
    if len(emotion_preview) == 1:
        axes = [axes]
    for axis, (class_name, image_path) in zip(axes, emotion_preview):
        with Image.open(image_path) as image:
            axis.imshow(image.convert('RGB'))
        axis.set_title(class_name)
        axis.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Class previews will appear here after the dataset is added.')

### LandingLens experiment

I would create a **Classification** project in LandingLens for the expression images. I would review a sample from every class before training and make sure one class does not contain only one lighting condition or background. After training, I would test new images and compare which expressions the model confuses. If `Surprised` and `Happy` are often mixed up, I would add more varied examples for both classes and check the labels again.

## Evaluation and conclusion

For the medicine assistant, I would check whether the model places boxes around the bottle and label on new photos. For the expression assistant, I would check its class-wise performance and review errors across different lighting and camera angles. The LandingLens training and test scores should be recorded after running each project in the platform. I have not entered scores here because they depend on the images and the actual training run.

LandingLens can provide the vision model and a deployable endpoint. The medicine assistant still needs OCR to read the label, and the expression classifier should only provide an optional cue that the user can correct.